# Mutual Fund Analytics Capstone

## Day 1: Data Ingestion

Objective:
Load all provided datasets, perform initial validation, and fetch live NAV data using the mfapi.in API.

In [6]:
import pandas as pd
from pathlib import Path
from pathlib import Path

pd.set_option("display.max_columns", None)

raw_path = Path("../data/raw")
processed_path = Path("../data/processed")

processed_path.mkdir(exist_ok=True)

print("Environment Ready")

Environment Ready


In [7]:
fund_master = pd.read_csv(raw_path / "01_fund_master.csv")
nav_history = pd.read_csv(raw_path / "02_nav_history.csv")
aum = pd.read_csv(raw_path / "03_aum_by_fund_house.csv")
sip = pd.read_csv(raw_path / "04_monthly_sip_inflows.csv")
category = pd.read_csv(raw_path / "05_category_inflows.csv")
folio = pd.read_csv(raw_path / "06_industry_folio_count.csv")
performance = pd.read_csv(raw_path / "07_scheme_performance.csv")
transactions = pd.read_csv(raw_path / "08_investor_transactions.csv")
holdings = pd.read_csv(raw_path / "09_portfolio_holdings.csv")
benchmark = pd.read_csv(raw_path / "10_benchmark_indices.csv")

print("All datasets loaded successfully")

All datasets loaded successfully


In [8]:
print(fund_master.shape)
print(nav_history.shape)
...

(40, 15)
(46000, 3)


Ellipsis

In [10]:
datasets = {
    "fund_master": fund_master,
    "nav_history": nav_history,
    "aum": aum,
    "sip": sip,
    "category": category,
    "folio": folio,
    "performance": performance,
    "transactions": transactions,
    "holdings": holdings,
    "benchmark": benchmark
}

for name, df in datasets.items():
    print(f"{name}: {df.duplicated().sum()} duplicate rows")

fund_master: 0 duplicate rows
nav_history: 0 duplicate rows
aum: 0 duplicate rows
sip: 0 duplicate rows
category: 0 duplicate rows
folio: 0 duplicate rows
performance: 0 duplicate rows
transactions: 0 duplicate rows
holdings: 0 duplicate rows
benchmark: 0 duplicate rows


In [11]:
for name, df in datasets.items():
    print(df.isnull().sum())

amfi_code             0
fund_house            0
scheme_name           0
category              0
sub_category          0
plan                  0
launch_date           0
benchmark             0
expense_ratio_pct     0
exit_load_pct         0
min_sip_amount        0
min_lumpsum_amount    0
fund_manager          0
risk_category         0
sebi_category_code    0
dtype: int64
amfi_code    0
date         0
nav          0
dtype: int64
date              0
fund_house        0
aum_lakh_crore    0
aum_crore         0
num_schemes       0
dtype: int64
month                         0
sip_inflow_crore              0
active_sip_accounts_crore     0
new_sip_accounts_lakh         0
sip_aum_lakh_crore            0
yoy_growth_pct               12
dtype: int64
month               0
category            0
net_inflow_crore    0
dtype: int64
month                  0
total_folios_crore     0
equity_folios_crore    0
debt_folios_crore      0
hybrid_folios_crore    0
others_folios_crore    0
dtype: int64
amfi_code

In [12]:
fund_master["fund_house"].unique()
fund_master["category"].unique()
fund_master["risk_category"].unique()

<StringArray>
['Moderate', 'Very High', 'Low', 'High', 'Moderately High']
Length: 5, dtype: str

In [16]:
import pandas as pd
import requests
from pathlib import Path

# Create output folder
output_folder = Path("data/raw/live_nav")
output_folder.mkdir(parents=True, exist_ok=True)

# Scheme codes
schemes = {
    "HDFC_Top_100": 125497,
    "SBI_Bluechip": 119551,
    "ICICI_Bluechip": 120503,
    "Nippon_Large_Cap": 118632,
    "Axis_Bluechip": 119092,
    "Kotak_Bluechip": 120841
}

for scheme_name, scheme_code in schemes.items():
    try:
        url = f"https://api.mfapi.in/mf/{scheme_code}"

        response = requests.get(url)
        response.raise_for_status()

        data = response.json()

        nav_df = pd.DataFrame(data["data"])

        output_file = output_folder / f"{scheme_name}.csv"
        nav_df.to_csv(output_file, index=False)

        print(f"SUCCESS: {scheme_name} saved")
        print(f"Rows: {len(nav_df)}")
        print("-" * 50)

    except Exception as e:
        print(f"ERROR fetching {scheme_name}: {e}")

print("\nLive NAV fetch completed.")

SUCCESS: HDFC_Top_100 saved
Rows: 3092
--------------------------------------------------
SUCCESS: SBI_Bluechip saved
Rows: 3237
--------------------------------------------------
SUCCESS: ICICI_Bluechip saved
Rows: 3308
--------------------------------------------------
SUCCESS: Nippon_Large_Cap saved
Rows: 3299
--------------------------------------------------
SUCCESS: Axis_Bluechip saved
Rows: 3566
--------------------------------------------------
SUCCESS: Kotak_Bluechip saved
Rows: 3302
--------------------------------------------------

Live NAV fetch completed.
